In [2]:
import pandas as pd
import numpy as np
from hmmlearn import hmm
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [3]:
data = pd.read_csv("../processed_data/training_features.csv")

print("Dataset shape:", data.shape)
print("Activities:", data['activity'].unique())

data.head()

Dataset shape: (3839, 27)
Activities: ['jumping' 'standing' 'still' 'walking']


,accel_x_mean,accel_x_std,accel_x_var,accel_y_mean,accel_y_std,accel_y_var,accel_z_mean,accel_z_std,accel_z_var,gyro_x_mean,...,gyro_z_var,accel_sma,accel_xy_corr,accel_xz_corr,accel_yz_corr,accel_x_dom_freq,accel_y_dom_freq,accel_z_dom_freq,activity,recording
0,-0.837983,1.758604,3.092689,0.171601,0.772426,0.596642,0.995800,2.133551,4.552041,-0.855934,...,0.444917,3.767699,0.378335,0.393158,0.446090,0,1,1,jumping,jump11-ange-2026-03-06_09-59-13
1,0.014290,2.361475,5.576565,-0.503273,1.993134,3.972584,-2.278114,5.302432,28.115784,-0.392847,...,0.349308,7.963501,-0.174241,-0.030494,0.498210,2,1,1,jumping,jump11-ange-2026-03-06_09-59-13
2,0.991794,1.638522,2.684755,-0.887473,1.829146,3.345774,-0.239770,7.099894,50.408498,-0.123881,...,0.155363,9.317532,-0.063957,0.238329,0.440889,0,0,1,jumping,jump11-ange-2026-03-06_09-59-13
3,0.655566,1.215642,1.477785,-0.893239,1.913605,3.661884,0.307948,6.883720,47.385596,0.306612,...,0.380792,8.921468,-0.231635,0.285444,0.535261,1,0,1,jumping,jump11-ange-2026-03-06_09-59-13
4,-0.018417,1.012740,1.025643,-0.530605,2.308746,5.330307,0.093246,6.671704,44.511641,0.076820,...,0.144026,8.833271,-0.344359,-0.027778,0.675745,2,1,1,jumping,jump11-ange-2026-03-06_09-59-13


In [4]:
X = data.drop(columns=["activity", "recording"])
y = data["activity"]
recordings = data["recording"]

In [5]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 3071
Testing samples: 768


In [7]:
activities = y.unique()
models = {}

for activity in activities:

    X_activity = X_train[y_train == activity]

    model = hmm.GaussianHMM(
        n_components=4,        # number of hidden states
        covariance_type="diag",
        n_iter=100
    )

    model.fit(X_activity)

    models[activity] = model

print("Trained models for:", list(models.keys()))

Model is not converging.  Current: 39519.74553992406 is not greater than 39519.789070114624. Delta is -0.04353019056725316


Trained models for: ['jumping', 'standing', 'still', 'walking']


In [8]:
def predict_activity(sample):

    scores = {}

    for activity, model in models.items():
        score = model.score(sample)
        scores[activity] = score

    return max(scores, key=scores.get)

In [9]:
predictions = []

for i in range(len(X_test)):
    sample = X_test.iloc[i:i+1]
    pred = predict_activity(sample)
    predictions.append(pred)

predictions = np.array(predictions)

In [10]:
from sklearn.metrics import accuracy_score, classification_report

accuracy = accuracy_score(y_test, predictions)

print("Accuracy:", accuracy)
print(classification_report(y_test, predictions))

Accuracy: 0.7252604166666666
              precision    recall  f1-score   support

     jumping       0.75      0.98      0.85       200
    standing       0.59      0.93      0.73       208
       still       0.92      0.25      0.39       177
     walking       0.92      0.67      0.77       183

    accuracy                           0.73       768
   macro avg       0.80      0.71      0.69       768
weighted avg       0.79      0.73      0.69       768



In [11]:
import joblib

joblib.dump(models, "../models/hmm_models.pkl")
joblib.dump(scaler, "../models/scaler.pkl")

print("Models saved successfully!")

Models saved successfully!
